# ISIC2024 후속 검증 Tabular 결과 분석 17

이 notebook은 `.py` runner가 artifact를 만든 뒤에 후속 tabular baseline 결과를 읽고 비교하는 분석 문서다.

- 주 평가지표: `pauc_above_tpr80`
- 비교 모델: `xgboost`, `catboost`, `svm`, `logistic_regression`, `mlp`
- feature set 초점: `strict`, `relaxed`, `oracle`
- 실행 기준 원본: `.py` runner와 그 결과 artifact


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("/home/junkim2603a/proj/paper_ajou_dev").resolve()
os.chdir(ROOT)

TABULAR_LEADERBOARD_CANDIDATES = [
    ROOT / "artifacts" / "tabular" / "mlflow_leaderboard.csv",
    ROOT / "artifacts" / "mlflow_leaderboard.csv",
]
TABULAR_RUN_ROOT = ROOT / "artifacts" / "tabular_runs"
EXPORT_DIR = ROOT / "artifacts" / "eda" / "isic2024" / "followup_tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def find_primary_metric(columns: list[str], prefix: str) -> str | None:
    exact = [column for column in columns if column == prefix]
    if exact:
        return exact[0]
    candidates = sorted(column for column in columns if column.startswith(prefix))
    return candidates[0] if candidates else None


def load_parent_leaderboard() -> tuple[pd.DataFrame, Path | None]:
    for path in TABULAR_LEADERBOARD_CANDIDATES:
        if path.exists():
            return pd.read_csv(path), path
    return pd.DataFrame(), None


def load_trial_summaries() -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    if not TABULAR_RUN_ROOT.exists():
        return pd.DataFrame()
    for summary_path in sorted(TABULAR_RUN_ROOT.glob("*/*/summary.json")):
        payload = read_json(summary_path)
        metrics = payload.get("metrics", {})
        hyperparameters = payload.get("hyperparameters", {})
        rows.append(
            {
                "summary_path": str(summary_path),
                "model_name": payload.get("model_name", summary_path.parts[-3]),
                "feature_set": hyperparameters.get("feature_set", ""),
                "val_pauc_above_tpr80": metrics.get("val", {}).get("pauc_above_tpr80"),
                "test_pauc_above_tpr80": metrics.get("test", {}).get("pauc_above_tpr80"),
                "test_average_precision": metrics.get("test", {}).get("average_precision"),
                "test_auc_roc": metrics.get("test", {}).get("auc_roc"),
            }
        )
    return pd.DataFrame(rows)


input_status = pd.DataFrame(
    [
        {"path": str(path), "exists": path.exists()}
        for path in [*TABULAR_LEADERBOARD_CANDIDATES, TABULAR_RUN_ROOT]
    ]
)
display(input_status)


In [ ]:
leaderboard_df, leaderboard_path = load_parent_leaderboard()
trial_df = load_trial_summaries()

print("leaderboard 경로:", leaderboard_path)
print("leaderboard 행 수:", len(leaderboard_df))
print("trial 행 수:", len(trial_df))

if leaderboard_df.empty and trial_df.empty:
    display(
        Markdown(
            "아직 tabular 결과가 없습니다. 먼저 tabular benchmark를 실행한 뒤 이 notebook을 다시 여세요."
        )
    )
else:
    if not leaderboard_df.empty:
        primary_column = find_primary_metric(list(leaderboard_df.columns), "best_pauc_above_tpr")
        sort_columns = [primary_column] if primary_column else []
        visible = leaderboard_df.sort_values(sort_columns, ascending=False) if sort_columns else leaderboard_df.copy()
        display(visible)
    if not trial_df.empty:
        display(trial_df.sort_values(["feature_set", "test_pauc_above_tpr80"], ascending=[True, False]))


In [ ]:
if trial_df.empty:
    display(Markdown("아직 trial 단위 tabular summary가 없어 비교 차트는 건너뜁니다."))
else:
    pivot_df = (
        trial_df.groupby(["feature_set", "model_name"], dropna=False)["test_pauc_above_tpr80"]
        .max()
        .reset_index()
        .sort_values(["feature_set", "test_pauc_above_tpr80"], ascending=[True, False])
    )
    display(pivot_df)

    fig, ax = plt.subplots(figsize=(12, 5))
    for feature_set in pivot_df["feature_set"].dropna().unique():
        subset = pivot_df[pivot_df["feature_set"] == feature_set]
        ax.plot(subset["model_name"], subset["test_pauc_above_tpr80"], marker="o", label=feature_set)
    ax.set_title("Tabular 후속 검증 Test pAUC@TPR>=0.80")
    ax.set_ylabel("test_pauc_above_tpr80")
    ax.set_xlabel("model_name")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
if leaderboard_df.empty:
    print("tabular leaderboard가 없어 export를 건너뜁니다.")
else:
    export_path = EXPORT_DIR / "tabular_followup_parent_leaderboard.csv"
    leaderboard_df.to_csv(export_path, index=False)
    print("저장 완료:", export_path)
